# 06 - Machine Learning Churn Prediction & SHAP Explainability
**Platform:** Bangladesh E-Commerce Customer Intelligence Platform  
**Focus:** Leakage-Free Temporal Split, Model Comparison, and Explainable AI (SHAP)

---
## Business Definition of Churn
In non-contractual e-commerce, customers do not explicitly cancel subscriptions. We define **Churn** as:
> **Zero delivered purchase events within an active 90-day observation window.**

### Strict Avoidance of Data Leakage
- **Observation Window:** Last 90 days of the dataset (2025-07-02 to 2025-09-30).
- **Feature Window:** Strict historical data up to and including the Cutoff Date (<= 2025-07-02).
All behavioral features (Recency, Frequency, Spend Momentum, Discount Ratio) are engineered exclusively using data available before the cutoff date.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

from src.analytics.eda import load_curated_data
from src.churn.model import create_leakage_free_churn_dataset, train_and_compare_models
from src.churn.explainability import compute_shap_explanations, explain_individual_customer

df_cust, df_prod, df_ord, df_items = load_curated_data()
print("Data loaded.")

## 1. Construct Leakage-Free Dataset

In [ ]:
X, y, full_churn_df = create_leakage_free_churn_dataset(df_cust, df_ord, df_items, df_prod, cutoff_days=90)
print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

## 2. Train & Compare Machine Learning Models
- Logistic Regression (Baseline)
- Random Forest
- XGBoost (Gradient Boosted Trees)

In [ ]:
model_results, best_model, scaler, test_data = train_and_compare_models(X, y)
df_comp = pd.DataFrame(model_results).T
display(df_comp)

## 3. Global Model Explainability via SHAP

In [ ]:
# Compute SHAP values on a test sample for computational efficiency
X_sample = test_data['X_test'].sample(n=min(1000, len(test_data['X_test'])), random_state=42)
shap_vals, explainer, importance_df = compute_shap_explanations(best_model, X_sample)

print("Top 10 Global Churn Drivers:")
display(importance_df.head(10))

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'].head(10)[::-1], importance_df['mean_abs_shap'].head(10)[::-1], color='#1f77b4')
plt.title('Global Feature Importance (Mean |SHAP Value|)', fontsize=13, fontweight='bold')
plt.xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.show()

## 4. Individual Customer Case Diagnostic
Let's explain why a specific high-risk customer is predicted to churn.

In [ ]:
high_risk_idx = test_data['y_test'][test_data['y_test'] == 1].index[0]
sample_cust = X.loc[high_risk_idx]
cust_id = full_churn_df.loc[high_risk_idx, 'customer_id']

diag = explain_individual_customer(sample_cust, best_model, explainer, feature_names=list(X.columns))
print(f"Customer ID: CUST-{cust_id}")
print(f"Predicted Churn Probability: {diag['churn_probability'] * 100:.1f}%")
print(f"Risk Tier: {diag['predicted_risk_tier']}")
print("\nTop Drivers:")
for d in diag['top_drivers']:
    print(f" - {d['feature']} (Value: {d['value']}): SHAP impact = {d['shap_impact']:.4f} [{d['direction']}]")